In [2]:
import json
import os
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import requests

from vla_verify.verifier import VLAVerifier
from vla_verify.scene_graph import TaskSceneGraph
from vlm_interfaces import *

In [ ]:
def run_skill_pddl(name, params, image):
    hack_skill = f"{name.upper()}({', '.join(params)})"
    print(hack_skill)
    pddl_action = scene_graph.match_grounded_action(name, params)
    if pddl_action is None:
        print("Could not match pddl action!")
        return None, None
    target = None
    match pddl_action.name.value:
        case "pickup_from" | "open" | "close" | "turn_on" | "turn_off":
            target = pddl_action.grounding[0].value
        case "place_on" | "place_in" :
            target = pddl_action.grounding[2].value
            # TODO: table location is bad

    if target in scene_graph.object_data:
        print(f"Grounding object {target}")
        target_object = scene_graph.object_data[target]
        target_info = target_object.to_dict(include_grounding=False)
        result = scene_graph.ground_openrouter(image, target_object, task_hint=hack_skill)
        print("Grounding result:", result)
        if result['status'] == 'OK':
            target_info['image_point'] = result['position'].tolist()
    else:
        print(f"Object {target} not found in scene graph!")
        target_info = None
    return target_info, pddl_action

In [ ]:
PDDL_PATH = "pddl/libero_domain.pddl"

llm_interface, vlm_interface = get_openrouter_interfaces()
pddl_domain_text = open(PDDL_PATH).read()
scene_graph = TaskSceneGraph(pddl_domain_text, vlm_interface)

In [ ]:
def get_image():
    !scp signal@10.89.50.26:~/workspace/hardware_setup/i2rt/data_collection/cam1_latest.png scene_graph_input.png
    return cv2.cvtColor(cv2.imread("scene_graph_input.png"), cv2.COLOR_BGR2RGB)

In [ ]:
image = get_image()
plt.figure(0)
plt.clf()
plt.imshow(image)

In [ ]:
scene_graph.read_image(image, ground=False, hint="The robot is only trying to manipulate things on the tabletop.")

In [ ]:
from llm_apis import transformers_api
from llm_apis.response_parsing import extract_in_backticks, extract_json_from_response

def _get_pddl_action(llm_response, scene_graph, task, image_rgb):
    user_prompt = GET_PDDL_ACTION_USER_PROMPT.format(
        pddl_scene=scene_graph.pddl_summary(),
        user_task=task
    )
    yield [ transformers_api.make_message(texts=user_prompt, images=[image_rgb]) ]

    raw_response = llm_response['content']
    try:
        result = extract_json_from_response(raw_response)
        yield result
    except Exception as e:
        print(f"Warning: VLM response extraction raised exception: {e}")
        yield { "status": "LLM_ERROR" }
    
_get_pddl_action.system_prompt = f"""\
The following is a PDDL domain description for a generic pick and place task:

```pddl
{pddl_domain_text}
```

The user is trying to execute a task, given as a natural language prompt.

Given an image of the world and a description of the current scene state, return \
a PDDL action that makes the most sense to take to make progress towards the goal.

The natural language prompt may not match the scene graph descriptions perfectly. \
You should allow for minor mismatches in descriptions of objects, such as slightly \
different colors.

Object descriptions in the scene description are formatted as PDDL comments:
<object id> - <object PDDL type> ; <object appearance> | <object location>

Give your response in the following format:

```json
{{
  "status": OK | ERROR,
  "reasoning": <text explanation for your decision>,
  "action_name": <name of the pddl action to take>,
  "action_params": [
      <list of the action parameters, as strings.>
  ]
}}
```

`action_name` and `action_params` can be omitted if the task is impossible (status should be set to ERROR).

"""

GET_PDDL_ACTION_USER_PROMPT = """\
Task:
{user_task}

Scene state:
{pddl_scene}
"""
get_pddl_action = vlm_interface(_get_pddl_action)

In [ ]:
image = get_image()
plt.figure(0)
plt.clf()
plt.imshow(image)
plt.show()
resp = get_pddl_action(scene_graph, "put red and orange vegetables into the red plate", image)
print(resp)

In [ ]:
res, pddl_action = run_skill_pddl(resp['action_name'], resp['action_params'], image)
print(res)

if res is not None:
    plt.figure(0)
    plt.clf()
    plt.imshow(image)
    plt.scatter(*(res['image_point'] * np.array(image.shape[:2][::-1])), color='red')
    plt.show()
    input("Press enter when done with action...")
    scene_graph.apply_action(pddl_action, new_image=get_image())

In [ ]:
print(scene_graph.pddl_summary())